# Raw API Calls

Direct `requests` calls to each API — **no collector classes, no database**.
Just the HTTP request and the raw JSON it returns, so you can see exactly what each source gives you.

Run the **imports** cell first, then any source cell.

In [9]:
# === Imports (run once) ===
import requests
import json

def show(data, n=2):
    """Pretty-print the first n records of a JSON response."""
    print(json.dumps(data[:n] if isinstance(data, list) else data, indent=2)[:20000])

## Senate LDA — lobbying filings (no key)

Endpoint: `https://lda.senate.gov/api/v1/filings/`

In [10]:
# === Senate LDA ===
resp = requests.get(
    "https://lda.senate.gov/api/v1/filings/",
    params={"filing_year": 2024, "page": 15, "page_size": 15},
    timeout=30,
)
print("Status:", resp.status_code)
data = resp.json()
print("Total available:", data["count"])
show(data["results"])

Status: 200
Total available: 96854
[
  {
    "url": "https://lda.senate.gov/api/v1/filings/c41a6f36-9b5a-441f-a6c3-bf85c7c939be/",
    "filing_uuid": "c41a6f36-9b5a-441f-a6c3-bf85c7c939be",
    "filing_type": "RR",
    "filing_type_display": "Registration",
    "filing_year": 2024,
    "filing_period": "first_quarter",
    "filing_period_display": "1st Quarter (Jan 1 - Mar 31)",
    "filing_document_url": "https://lda.senate.gov/filings/public/filing/c41a6f36-9b5a-441f-a6c3-bf85c7c939be/print/",
    "filing_document_content_type": "text/html",
    "income": null,
    "expenses": null,
    "expenses_method": null,
    "expenses_method_display": null,
    "posted_by_name": "Richard Grafmeyer",
    "dt_posted": "2024-02-01T11:06:58-05:00",
    "termination_date": null,
    "registrant_country": "United States of America",
    "registrant_ppb_country": null,
    "registrant_address_1": "101 Constitution Avenue, NW Suite 675 East",
    "registrant_address_2": null,
    "registrant_different

## FEC — committees / Super PACs (DEMO_KEY)

Endpoint: `https://api.open.fec.gov/v1/committees`

In [3]:
# === FEC committees ===
resp = requests.get(
    "https://api.open.fec.gov/v1/committees",
    params={"api_key": "DEMO_KEY", "committee_type": "O", "per_page": 5, "page": 1},
    timeout=30,
)
print("Status:", resp.status_code)
data = resp.json()
print("Total available:", data["pagination"]["count"])
show(data["results"])

Status: 200
Total available: 8524
[
  {
    "affiliated_committee_name": "NONE",
    "candidate_ids": [],
    "committee_id": "C00877886",
    "committee_type": "O",
    "committee_type_full": "Super PAC (Independent Expenditure-Only)",
    "cycles": [
      2024,
      2026
    ],
    "designated_agent_city": "TALLAHASSEE",
    "designated_agent_first_name": "SHELBY",
    "designated_agent_last_name": "GREEN",
    "designated_agent_middle_name": null,
    "designated_agent_name": "GREEN, SHELBY",
    "designated_agent_phone_number": "8506613941",
    "designated_agent_prefix": null,
    "designated_agent_state": "FL",
    "designated_agent_street1": "2800 S ADAMS ST UNIT 5651",
    "designated_agent_street2": null,
    "designated_agent_suffix": null,
    "designated_agent_title": "TREASURER",
    "designated_agent_zip": "32314",
    "designation": "U",
    "designation_full": "Unauthorized",
    "filing_frequency": "Q",
    "first_f1_date": "2024-05-02",
    "first_file_date": "2024-

## FEC — disbursements / spending (DEMO_KEY)

Endpoint: `https://api.open.fec.gov/v1/schedules/schedule_b`

In [11]:
# === FEC disbursements ===
resp = requests.get(
    "https://api.open.fec.gov/v1/schedules/schedule_b",
    params={"api_key": "DEMO_KEY", "two_year_transaction_period": 2024, "per_page": 5, "page": 1},
    timeout=30,
)
print("Status:", resp.status_code)
data = resp.json()
show(data["results"])

Status: 200
[
  {
    "amendment_indicator": "A",
    "amendment_indicator_desc": "ADD",
    "back_reference_schedule_id": "SB17",
    "back_reference_transaction_id": "B664BDFCC4E9C439A854",
    "beneficiary_committee_name": null,
    "candidate_first_name": null,
    "candidate_id": null,
    "candidate_last_name": null,
    "candidate_middle_name": null,
    "candidate_name": null,
    "candidate_office": null,
    "candidate_office_description": null,
    "candidate_office_district": null,
    "candidate_office_state": null,
    "candidate_office_state_full": null,
    "candidate_prefix": null,
    "candidate_suffix": null,
    "category_code": null,
    "category_code_full": null,
    "comm_dt": null,
    "committee": {
      "affiliated_committee_name": "TEAM CELESTE",
      "candidate_ids": [
        "H4UT02296"
      ],
      "city": "CEDAR CITY",
      "committee_id": "C00842765",
      "committee_type": "H",
      "committee_type_full": "House",
      "cycle": 2024,
      "cy

## Congress.gov — bills (needs a key)

Get a free key at https://api.congress.gov/sign-up/ and paste it below.
Endpoint: `https://api.congress.gov/v3/bill/{congress}/{type}`

In [12]:
# === Congress.gov bills ===
CONGRESS_API_KEY = "PASTE_YOUR_KEY_HERE"

resp = requests.get(
    "https://api.congress.gov/v3/bill/118/hr",
    params={"api_key": CONGRESS_API_KEY, "format": "json", "limit": 5},
    timeout=30,
)
print("Status:", resp.status_code)
data = resp.json()
show(data["bills"])

Status: 403


KeyError: 'bills'

# Amendment text — and the Congressional Record

**The key thing to understand before doing NLP on amendments:**

The amendment text endpoint is `/v3/amendment/{congress}/{type}/{number}/text`, but coverage is uneven:

| Case | What you get |
|------|--------------|
| **Senate amendments, 117th Congress (2021) → now** | Full text directly in the API (`textVersions` → HTML/PDF) ✅ clean |
| House amendments (any congress) | **Only a link into the Congressional Record** — no text field |
| Senate amendments **before** the 117th | **Only a link into the Congressional Record** |
| Some House amendments 117th+ | No text granule available at all |

So there are **two paths** to amendment text:
1. **API-native** (easy, clean) — only Senate amendments 117th+.
2. **Congressional Record** (harder) — page-ranged granules (e.g. `S1234`–`S1240`) whose actual text lives as HTML/PDF on GovInfo.

**Scoping implication:** for the capstone, the clean spine is *Senate amendments, 117th Congress forward*. Older / House amendment text means scraping the Record and is a stretch goal.

> Set your key in the next cell to run these.

In [ ]:
# === Amendment key + helper ===
# Paste your key here (same one as the bills cell).
CONGRESS_API_KEY = ""
BASE = "https://api.congress.gov/v3"

def cg_get(path, **params):
    """GET a Congress.gov path and return JSON."""
    params = {"api_key": CONGRESS_API_KEY, "format": "json", **params}
    r = requests.get(f"{BASE}/{path.lstrip('/')}", params=params, timeout=30)
    r.raise_for_status()
    return r.json()

### Step 1 — list Senate amendments (117th), pick one

In [16]:
# === Step 1: list amendments ===
# SAMDT = Senate amendment. 117th Congress is the first with full text.
data = cg_get("amendment/117/samdt", limit=5)
for a in data["amendments"]:
    print(a["type"], a["number"], "-", a.get("purpose") or a.get("description") or "(no purpose)")

SAMDT 2 - (no purpose)
SAMDT 3 - (no purpose)
SAMDT 4 - (no purpose)
SAMDT 5 - (no purpose)
SAMDT 1 - In the nature of a substitute.


### Step 2 — get the text endpoint for one amendment

This returns `textVersions` → `formats` (HTML / PDF). For Senate 117th+ these are real text files.

In [17]:
# === Step 2: amendment text endpoint ===
# Use a known Senate amendment with full text. SA 2137 (117th) on the Infrastructure bill.
CONGRESS, AMDT_TYPE, AMDT_NUM = 117, "samdt", 2137

text_data = cg_get(f"amendment/{CONGRESS}/{AMDT_TYPE}/{AMDT_NUM}/text")
versions = text_data.get("textVersions", [])
print(f"Text versions available: {len(versions)}\n")
for v in versions:
    print("type:", v.get("type"), "| date:", v.get("date"))
    for fmt in v.get("formats", []):
        print("   ", fmt.get("type"), "->", fmt.get("url"))

Text versions available: 1

type: Submitted | date: 2021-08-01T04:00:00Z
    HTML -> https://www.congress.gov/117/crec/2021/08/01/167/136/modified/CREC-2021-08-01-pt1-PgS5255.htm
    PDF -> https://www.congress.gov/117/crec/2021/08/01/167/136/CREC-2021-08-01-pt1-PgS5255.pdf


### Step 3 — download the actual text

Grab the HTML (or PDF) URL from above and fetch it. This is the raw amendment text you'd feed into NLP.

In [18]:
# === Step 3: download the text ===
# Pick the first HTML format URL we found above.
html_url = None
for v in versions:
    for fmt in v.get("formats", []):
        if fmt.get("type") in ("HTML", "Formatted Text"):
            html_url = fmt["url"]
            break
    if html_url:
        break

if html_url:
    raw = requests.get(html_url, timeout=30).text
    print("Fetched", len(raw), "chars from:", html_url, "\n")
    print(raw[:1500])
else:
    print("No HTML format on this amendment — it likely only links to the Congressional Record.")

Fetched 3086081 chars from: https://www.congress.gov/117/crec/2021/08/01/167/136/modified/CREC-2021-08-01-pt1-PgS5255.htm 

<pre>


[Pages S5255-S5528]
From the Congressional Record Online through the Government Publishing Office [<a href='https://www.gpo.gov'>www.gpo.gov</a>]

  SA 2137. Mr. SCHUMER (for Ms. Sinema (for herself, Mr. Portman, Mr. 
Manchin, Mr. Cassidy, Mrs. Shaheen, Ms. Collins, Mr. Tester, Ms. 
Murkowski, Mr. Warner, and Mr. Romney)) proposed an amendment to the 
bill H.R. 3684, to authorize funds for Federal-aid highways, highway 
safety programs, and transit programs, and for other purposes; as 
follows:

        Strike all after the enacting clause and insert the 
     following:

     SECTION 1. SHORT TITLE; TABLE OF CONTENTS.

       (a) Short Title.--This Act may be cited as the 
     ``Infrastructure Investment and Jobs Act&#x27;&#x27;.
       (b) Table of Contents.--The table of contents for this Act 
     is as follows:

Sec. 1. Short title; table of contents

### Step 4 — the Congressional Record fallback (House / pre-117th)

When an amendment has **no `textVersions`**, its text only exists in the Congressional Record.
You navigate the Record by **volume → issue → articles**, and each article has page ranges
(`startPage`/`endPage`, e.g. `S1234`) plus text URLs (HTML/PDF granules hosted on GovInfo).

The amendment's `latestAction.text` usually names the page (e.g. "Submitted ... see Congressional Record S1234"),
which you match against article page ranges to find the right granule.

In [19]:
# === Step 4: navigate the Congressional Record ===
# List recent issues to see the volume/issue structure, then drill into one issue's articles.
issues = cg_get("daily-congressional-record", limit=3)
for iss in issues["dailyCongressionalRecord"]:
    print("Vol", iss["volumeNumber"], "Issue", iss["issueNumber"], "-", iss["issueDate"])

# Drill into one issue's articles (each article = a chunk of the Record with page range + text URL).
vol = issues["dailyCongressionalRecord"][0]["volumeNumber"]
issue = issues["dailyCongressionalRecord"][0]["issueNumber"]
arts = cg_get(f"daily-congressional-record/{vol}/{issue}/articles")

print("\nArticles in Vol", vol, "Issue", issue, ":")
for section in arts.get("articles", []):
    print(" Section:", section.get("name"))
    for item in section.get("sectionArticles", [])[:3]:
        pages = f"{item.get('startPage')}-{item.get('endPage')}"
        url = (item.get("text") or [{}])[0].get("url")
        print("   ", pages, "|", item.get("title", "")[:60], "|", url)

Vol 172 Issue 99 - 2026-06-11T04:00:00Z
Vol 172 Issue 98 - 2026-06-10T04:00:00Z
Vol 172 Issue 97 - 2026-06-09T04:00:00Z

Articles in Vol 172 Issue 99 :
 Section: Daily Digest
    D619-D620 | Daily Digest/Next Meeting of the SENATE + Next Meeting of th | https://www.congress.gov/119/crec/2026/06/11/172/99/modified/CREC-2026-06-11-pt1-PgD619-3.htm
    D619-D619 | Daily Digest/COMMITTEE MEETINGS FOR 2026-06-15; Congressiona | https://www.congress.gov/119/crec/2026/06/11/172/99/modified/CREC-2026-06-11-pt1-PgD619-2.htm
    D619-D619 | Daily Digest/House Committee Meetings; Congressional Record  | https://www.congress.gov/119/crec/2026/06/11/172/99/modified/CREC-2026-06-11-pt1-PgD619.htm
 Section: Extensions of Remarks Section
    E571-E571 | RECOGNIZING THE POTOMAC SCHOOL ROBOTICS TEAMS FOR THEIR SUCC | https://www.congress.gov/119/crec/2026/06/11/172/99/modified/CREC-2026-06-11-pt1-PgE571-4.htm
    E571-E571 | RECOGNIZING LIVINGSTON "LIVIE" AND THALIA WALKER; Congressio | https://www.cong

# Text comparison — sentence-transformers + cosine similarity

Quick proof-of-concept for comparing two pieces of text by **meaning**, not exact words.

**How it works:**
1. Encode each text into an embedding vector with `sentence-transformers`.
2. Compute **cosine similarity** between the two vectors (1.0 = identical meaning, 0 = unrelated).
3. Compare the score to a threshold to decide "similar" vs "not similar".

We run several **success** cases (paraphrases that should match, unrelated texts that should not)
and a few **failure / edge** cases (e.g. negation) to show where this approach breaks down.

> First run downloads the `all-MiniLM-L6-v2` model (~90 MB).


In [2]:
# === Imports + model (run once) ===
# pip install sentence-transformers
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")  # small, fast, good baseline

# Cosine similarity comes from the library — no need to hand-roll it.
#   util.cos_sim(a, b)            -> sentence-transformers helper (returns a tensor)
#   sklearn.metrics.pairwise.cosine_similarity(a, b)  -> sklearn alternative


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# === Test cases: (label, text_a, text_b, expected_similar) ===
cases = [
    # --- SUCCESS: paraphrases that SHOULD match ---
    ("paraphrase_refund",   "You can request a full refund within 30 days.",
                            "A complete refund is available if you ask within thirty days.", True),
    ("paraphrase_meeting",  "Let's meet at 10 AM tomorrow.",
                            "Can we meet tomorrow at 10 in the morning?", True),
    ("paraphrase_shipping", "The package will arrive later because of weather.",
                            "Bad weather is causing a delay in delivery.", True),

    # --- SUCCESS: unrelated texts that SHOULD NOT match ---
    ("diff_music_finance",  "I love playing jazz piano on weekends.",
                            "The central bank increased interest rates.", False),
    ("diff_cooking_network","Boil pasta for eight minutes, then drain.",
                            "The router dropped packets during the stress test.", False),

    # --- FAILURE / EDGE: where embeddings tend to struggle ---
    ("edge_negation",       "The device is safe for children.",
                            "The device is not safe for children.", False),  # opposite meaning, high score
    ("edge_word_overlap",   "Apple stock surged today.",
                            "I ate an apple today.", False),                  # shared word, different topic
]


In [8]:
# === Run comparison + PASS/FAIL report ===
THRESHOLD = 0.55  # >= this score => predicted "similar"

texts_a = [c[1] for c in cases]
texts_b = [c[2] for c in cases]
emb_a = model.encode(texts_a, convert_to_tensor=True)
emb_b = model.encode(texts_b, convert_to_tensor=True)

# util.cos_sim returns an NxN matrix; we want the diagonal (pair i vs pair i).
scores = util.cos_sim(emb_a, emb_b).diagonal()

print(f"Threshold: {THRESHOLD}\n" + "-" * 78)
passed = 0
for (label, a, b, expected), score in zip(cases, scores):
    score = float(score)
    predicted = score >= THRESHOLD
    ok = predicted == expected
    passed += ok
    print(f"[{'PASS' if ok else 'FAIL'}] {label:<22} score={score:0.3f}  "
          f"expected={expected!s:<5} predicted={predicted}")

print("-" * 78)
print(f"Passed {passed}/{len(cases)}  |  Failed {len(cases) - passed}/{len(cases)}")


Threshold: 0.55
------------------------------------------------------------------------------
[PASS] paraphrase_refund      score=0.929  expected=True  predicted=True
[PASS] paraphrase_meeting     score=0.810  expected=True  predicted=True
[PASS] paraphrase_shipping    score=0.640  expected=True  predicted=True
[PASS] diff_music_finance     score=0.053  expected=False predicted=False
[PASS] diff_cooking_network   score=0.037  expected=False predicted=False
[FAIL] edge_negation          score=0.908  expected=False predicted=True
[PASS] edge_word_overlap      score=0.438  expected=False predicted=False
------------------------------------------------------------------------------
Passed 6/7  |  Failed 1/7


## Long texts (bill text, policy text)

The short-sentence demo above hides a real limit: **`all-MiniLM-L6-v2` only reads the first ~256 tokens (~200 words)** of any input. Encode a 5,000-word bill section and it silently keeps only the opening paragraph — the rest never reaches the model.

So a single embedding per document does **not** work for bills. Two practical fixes:

| Approach | How | When to use |
|----------|-----|-------------|
| **Chunk + aggregate** (below) | Split into ~150-word chunks, embed each, compare chunk-to-chunk | Default for bill/policy text with `all-MiniLM` |
| **Long-context model** | Use a model with a larger window, e.g. `BAAI/bge-m3` (8192 tokens) or `nlpaueb/legal-bert` for legal tone | When you want one vector per doc and have GPU/time |

For comparing bills, **max-pooling chunk similarities** answers "does *any* part of bill A closely match *any* part of bill B?" — which is usually what matters for detecting copied/lobbied language.


In [9]:
# === 1. Show the truncation problem ===
# The model's max input length is fixed; anything longer is cut off.
print("Max sequence length (word-pieces):", model.max_seq_length)

# A long block of policy-style text (repeat to simulate a multi-page bill section).
long_bill = (
    "SECTION 1. SHORT TITLE. This Act may be cited as the Clean Energy Investment Act. "
    "SEC. 2. The Secretary shall establish a grant program to fund solar and wind "
    "infrastructure in rural communities, prioritizing projects that create local jobs. "
) * 40  # ~ a few thousand words

n_words = len(long_bill.split())
print(f"Document length: {n_words} words (~{n_words * 1.3:.0f} tokens) -> far over the limit")
print("Only roughly the first ~200 words actually influence the single-vector embedding.")


Max sequence length (word-pieces): 256
Document length: 1560 words (~2028 tokens) -> far over the limit
Only roughly the first ~200 words actually influence the single-vector embedding.


In [3]:
# === 2. Chunk -> embed -> aggregate (the fix for long docs) ===
def chunk_text(text, words_per_chunk=150, overlap=30):
    """Split text into overlapping word-windows so we don't cut a sentence's meaning in half."""
    words = text.split()
    step = words_per_chunk - overlap
    chunks = [" ".join(words[i:i + words_per_chunk])
              for i in range(0, len(words), step)]
    return [c for c in chunks if c.strip()]

def doc_similarity(doc_a, doc_b, agg="max"):
    """Compare two long documents by chunking both and aggregating chunk-pair cosine scores.

    agg='max'  -> best-matching passage (good for 'is any section copied?')
    agg='mean' -> overall thematic closeness
    """
    chunks_a = chunk_text(doc_a)
    chunks_b = chunk_text(doc_b)
    emb_a = model.encode(chunks_a, convert_to_tensor=True)
    emb_b = model.encode(chunks_b, convert_to_tensor=True)
    sim_matrix = util.cos_sim(emb_a, emb_b)  # (len_a x len_b)
    if agg == "max":
        return float(sim_matrix.max()), len(chunks_a), len(chunks_b)
    return float(sim_matrix.mean()), len(chunks_a), len(chunks_b)

# Two policy docs that share a funding clause but differ elsewhere.
bill_a = (
    "SEC. 2. The Secretary shall establish a grant program to fund solar and wind "
    "infrastructure in rural communities, prioritizing projects that create local jobs. "
) * 20 + "SEC. 3. Reporting requirements shall be submitted annually to Congress. " * 20

bill_b = (
    "Appropriations for highway maintenance and bridge repair are authorized through 2030. "
) * 20 + (
    "The Department shall fund solar and wind infrastructure in rural communities, "
    "prioritizing projects that create local jobs. "
) * 10

unrelated = "This Act concerns the regulation of commercial fishing quotas in coastal waters. " * 40

score_related, na, nb = doc_similarity(bill_a, bill_b, agg="max")
score_unrelated, _, nu = doc_similarity(bill_a, unrelated, agg="max")

print(f"bill_a chunks={na}, bill_b chunks={nb}, unrelated chunks={nu}\n")
print(f"[bill_a vs bill_b]   max chunk similarity = {score_related:0.3f}  (shared funding clause -> high)")
print(f"[bill_a vs unrelated] max chunk similarity = {score_unrelated:0.3f}  (different topic -> low)")


bill_a chunks=6, bill_b chunks=4, unrelated chunks=4

[bill_a vs bill_b]   max chunk similarity = 0.770  (shared funding clause -> high)
[bill_a vs unrelated] max chunk similarity = 0.288  (different topic -> low)


## Large PDFs (bill / policy PDFs)

A PDF just adds an **extraction + cleanup step** in front of the chunk-embed-aggregate pipeline:

```
PDF bytes -> extract text -> clean -> chunk_text() -> model.encode() -> util.cos_sim()
```

**Picking an extractor** (all three are installed here):

| Library | Best for | Notes |
|---------|----------|-------|
| `pypdf` | Simple, well-formed digital PDFs | Pure Python, fast, no system deps |
| `pdfplumber` | Multi-column layouts, tables | Better column/word ordering |
| `pymupdf` (`fitz`) | Speed + tricky layouts | Fastest; great fallback |

**Things that bite you with real bill PDFs:**
- **Scanned / image-only PDFs** extract *nothing* — they need OCR (`pytesseract` + `pdf2image`). Always check `len(text)` per page.
- **Headers/footers/line numbers** (e.g. page markers, "S1234", margin line numbers in the Congressional Record) become noise — strip them before chunking.
- **Hyphenated line breaks** (`infra-\nstructure`) and hard-wrapped lines should be de-hyphenated/joined.
- Process **page-by-page** for big PDFs so memory stays flat, then chunk the cleaned full text.


In [4]:
# === PDF -> clean text helpers ===
# pip install pypdf   (or pdfplumber / pymupdf)
import re
from pypdf import PdfReader

def pdf_to_text(source):
    """Extract text from a PDF given a file path or raw bytes, page by page.

    Returns (full_text, per_page_char_counts). A page with ~0 chars usually means
    it's a scanned image -> you'd need OCR for that page.
    """
    import io
    reader = PdfReader(source if isinstance(source, str) else io.BytesIO(source))
    pages = [(p.extract_text() or "") for p in reader.pages]
    return "\n".join(pages), [len(p) for p in pages]

def clean_bill_text(text):
    """Strip common Congressional-PDF noise before chunking."""
    # join hyphenated line breaks:  infra-\nstructure -> infrastructure
    text = re.sub(r"-\n", "", text)
    # drop standalone line-number / page markers like 'S1234' or bare line numbers
    text = re.sub(r"^\s*[SHE]?\d{1,5}\s*$", " ", text, flags=re.MULTILINE)
    # collapse whitespace/newlines into single spaces
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def pdf_doc_similarity(pdf_a, pdf_b, agg="max"):
    """Full pipeline: two PDFs (path or bytes) -> cleaned -> chunked -> aggregated similarity."""
    text_a = clean_bill_text(pdf_to_text(pdf_a)[0])
    text_b = clean_bill_text(pdf_to_text(pdf_b)[0])
    return doc_similarity(text_a, text_b, agg=agg)   # reuses the chunk pipeline from above


In [5]:
# === Self-contained PDF demo (build sample PDFs in memory, then compare) ===
import fitz  # pymupdf, only used here to MAKE sample PDFs so the demo is runnable

def make_pdf_bytes(text, words_per_page=120):
    """Render text across multiple pages and return the PDF as bytes."""
    words = text.split()
    doc = fitz.open()
    for i in range(0, len(words), words_per_page):
        page = doc.new_page()
        page.insert_textbox(fitz.Rect(50, 50, 550, 750),
                            " ".join(words[i:i + words_per_page]), fontsize=11)
    data = doc.tobytes()
    doc.close()
    return data

pdf_a = make_pdf_bytes(bill_a)          # reuses bill_a / bill_b from the long-text demo
pdf_b = make_pdf_bytes(bill_b)
pdf_unrelated = make_pdf_bytes(unrelated)

# 1) Inspect extraction quality (catch scanned/empty pages).
text_a, page_chars = pdf_to_text(pdf_a)
print(f"Extracted {len(page_chars)} pages, chars/page = {page_chars}")
print(f"Total chars after extraction: {len(text_a)}")
print("Cleaned preview:", clean_bill_text(text_a)[:160], "...\n")

# 2) Run the full PDF -> similarity pipeline.
related, na, nb = pdf_doc_similarity(pdf_a, pdf_b, agg="max")
unrel, _, nu = pdf_doc_similarity(pdf_a, pdf_unrelated, agg="max")
print(f"[PDF_a vs PDF_b]   max chunk similarity = {related:0.3f}  (shared clause -> high)")
print(f"[PDF_a vs unrelated PDF] max chunk similarity = {unrel:0.3f}  (different topic -> low)")


Extracted 6 pages, chars/page = [799, 799, 799, 799, 863, 575]
Total chars after extraction: 4639
Cleaned preview: SEC. 2. The Secretary shall establish a grant program to fund solar and wind infrastructure in rural communities, prioritizing projects that create local jobs.  ...

[PDF_a vs PDF_b]   max chunk similarity = 0.770  (shared clause -> high)
[PDF_a vs unrelated PDF] max chunk similarity = 0.288  (different topic -> low)


### Real bills from GovInfo

GovInfo hosts every bill PDF at a predictable URL:

```
https://www.govinfo.gov/content/pkg/BILLS-{congress}{type}{number}{version}/pdf/BILLS-{congress}{type}{number}{version}.pdf
```

e.g. `BILLS-117hr3684enr` = 117th Congress, H.R. 3684, **enr**olled (the Infrastructure Investment & Jobs Act).
Versions: `ih`=introduced House, `rh`=reported House, `eh`=engrossed House, `es`=engrossed Senate, `enr`=enrolled (final).

Below we download two **versions of the same bill** (should be very similar) and one **unrelated bill** (should be low), then run the exact PDF pipeline above. No API key needed.


In [1]:
# === Download real bill PDFs from GovInfo and compare ===
import requests

def govinfo_bill_pdf(congress, bill_type, number, version):
    """Fetch a bill PDF from GovInfo as bytes. version: ih, rh, eh, es, enr, ..."""
    pkg = f"BILLS-{congress}{bill_type}{number}{version}"
    url = f"https://www.govinfo.gov/content/pkg/{pkg}/pdf/{pkg}.pdf"
    r = requests.get(url, headers={"User-Agent": "influence-network-demo"}, timeout=60)
    r.raise_for_status()
    return r.content, url

### ⚠️ Real-bill finding: boilerplate inflates max-pool similarity

Actual scores from the cell above (117th Congress):

| Comparison | max-pool | mean-pool |
|------------|---------|-----------|
| HR3684 introduced vs enrolled (**same bill**) | **0.943** | 0.361 |
| HR3684 enrolled vs HR5376 IRA (**different bill**) | **0.847** | 0.330 |

**The problem:** legislative boilerplate — "there are authorized to be appropriated such sums as may be necessary", definitions, effective-date and severability clauses — appears in *almost every* bill. `max` just finds the single most boilerplate-similar pair of chunks, so even unrelated bills score ~0.85. Max-pool alone is **not** a reliable "is this the same/copied text" signal for bills.

**What to do instead for the influence-network use case:**
- **Filter boilerplate** before embedding (drop enacting clauses, headers, standard definitions) — or down-weight chunks that match a "boilerplate bank".
- **Use top-k mean** (average of the few best chunk matches) instead of a single max, so one stock phrase can't carry the score.
- **Compare at the section/provision level**, not whole-bill, and report *which* provisions match — that's the actionable output for tracking lobbied language.
- For exact copied passages, pair embeddings with a lexical check (e.g. n-gram / MinHash overlap) — embeddings find *paraphrase*, lexical finds *verbatim*.
